## Lab 4: API Keys

### Part 1: Using an API Key to Access a Web Service

Create a PDF of a webpage (https://andrewbeatty1.pythonanywhere.com/bookviewer.html).
Get API Key from  html2pdf.com

In [3]:
import requests
import urllib.parse
import json
import base64
from github import Github
from config import apikeys as cfg

API key has been saved separately in a config file. This is a good practice to avoid hardcoding sensitive information in your code.

In [3]:
# Load the API key from the config file
apikey = cfg["htmltopdfkey"]

In [4]:
# Define the target URL and the API endpoint
targeturl = "https://andrewbeatty1.pythonanywhere.com/bookviewer.html"
apiurl = "https://api.html2pdf.app/v1/generate"

In [5]:
# Set up the parameters for the API request
params = {
    "url": targeturl,
    "apiKey": apikey
}
# encode the parameters
parsed_params = urllib.parse.urlencode(params)
# construct the full request URL
request_url = f"{apiurl}?{parsed_params}"

In [7]:
# Make the API request to generate the PDF
response = requests.get(request_url)

In [8]:
print(response.status_code)

200


Save the pdf as a file.

In [9]:
with open ("bookviewer.pdf", "wb") as f:
    f.write(response.content)

### Part 2: Get private data from GitHub using a Personal Access Token (PAT)

In [4]:
# Get API Key from GitHub
githubkey = cfg["aprivateone"]

In [5]:
# Define the GitHub API endpoint and the repository information
apiurl = "https://api.github.com/repos/AnnaLozenko/aprivateone"

In [6]:
# Set up the authentication using the Personal Access Token (PAT)
response = requests.get(apiurl, auth= ("token", githubkey))

In [11]:
# Check the response status code and print the repository information
print(response.status_code)
repoJSON = response.json()

200


If the API key is valid, the response status should be 200. Otherwise, you may get a 401 Unauthorized error if the API key is invalid or missing.

In [10]:
# save the repository information to a JSON file
with open ("repo.json", "w") as f:
    json.dump(repoJSON, f, indent=4)

## Part 3: Script that makes a change to a GitHub repository using the GitHub API and a Personal Access Token (PAT)

Important: To update a file in a GitHub repository using the API, you need to have the necessary permissions to access and modify the repository (read and write). Make sure that your Personal Access Token (PAT) has the appropriate scopes (permissions) to perform the update operation on the repository. If you do not have the required permissions, you may encounter a 403 Forbidden error when trying to update the file.

In [21]:
# Define the GitHub API endpoint for updating a file and the repository information.
# I will attempt to change the README.md file in the repository.
targeturl = "https://api.github.com/repos/AnnaLozenko/aprivateone/contents/README.md"

In [22]:
headers = {
    "Authorization": f"Bearer {githubkey}",
    "Accept": "application/vnd.github+json"
}

In [23]:
# Make the API request to update the README.md file
response = requests.get(targeturl, headers=headers)
print(response.status_code)

200


In [24]:
# To update a file on GitHub using the API, you need to provide the current SHA of the file you want to update. The SHA is a unique identifier for the file's current state. You can get the SHA from the response of the GET request to the file's endpoint.
sha = response.json()["sha"]

In [102]:
# gitHub API requires the content to be base64 encoded when updating a file. So we need to encode the new content before making the API request to update the README.md file.
new_content = "Hello there! This is a new content for the README.md file."
new_content_encoded = base64.b64encode(new_content.encode("utf-8")).decode("utf-8")

In [103]:
# Now we can make the API request to update the README.md file. We need to include the new content, the current SHA of the file, and a commit message in the request payload.
update_payload = {
    "message": "Update README.md file via API",
    "content": new_content_encoded,
    "sha": sha, # Get the current SHA of the file to be updated
    "branch" : "main"
}

In [104]:
# Make the API request to update the README.md file
put_response = requests.put(targeturl, headers=headers, json=update_payload)
print(put_response.status_code)

200


### Part 4: Using packages to interact with GitHub API

[PyGitHub](https://pygithub.readthedocs.io/en/latest/introduction.html) is a Python library that provides an easy-to-use interface for interacting with the GitHub API. It allows you to perform various operations on GitHub repositories, such as creating issues, managing pull requests, and updating files, without having to manually construct API requests.

In [34]:
g = Github(githubkey)

C:\Users\annal\AppData\Local\Temp\ipykernel_1652\3871077155.py:1: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g = Github(githubkey)


Get the clone URL of the repository using PyGitHub. This is a more convenient way to interact with the GitHub API compared to manually constructing API requests.

In [35]:
# Get the repository information using PyGitHub
repo = g.get_repo("AnnaLozenko/aprivateone")
print(repo.clone_url)

https://github.com/AnnaLozenko/aprivateone.git


Get the download URL of  file named `test.txt` from the repository using PyGitHub.

In [36]:
# Get the file information for `test.txt` using PyGitHub
fileInfo = repo.get_contents("test.txt")
url= fileInfo.download_url

Output the content of the file `test.txt` using the download URL obtained from PyGitHub. This allows you to access the content of the file directly without having to manually construct API requests.

In [37]:
# make the API request to download the content of `test.txt` using the download URL
response = requests.get(url)
print(response.status_code)

200


In [38]:
content = response.text
print(content)

In a city where the streetlights flickered like sleepy fireflies, there lived a young app developer named Andrew. By day, he fixed bugs and pushed updates. By night, he secretly believed the old stories his grandmother used to tell — the ones about hidden magic humming beneath ordinary things.

One evening, while debugging code at far too late an hour, Andrew noticed something strange on his screen. Between lines of perfectly normal Python, a message appeared:

// if you can read this, follow the blue cursor

Andrew blinked. “That… was not in the repo.”

The cursor began to move on its own, sliding across his monitor like it had somewhere very important to be. Against his better judgment (and every cybersecurity instinct he possessed), Andrew followed it.

Click.

Suddenly his apartment filled with a soft electric glow. His mechanical keyboard started typing by itself.

WELCOME, ANDREW. PATCH REQUIRED.

Now, Andrew had seen many weird production issues in his life, but none of them inv

Append more text to the file `test.txt` using PyGitHub. This demonstrates how to update a file in a GitHub repository using the PyGitHub library, which abstracts away the complexities of manually constructing API requests.

In [39]:
new_content = content + "\n\nAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA\n"
print(new_content)

In a city where the streetlights flickered like sleepy fireflies, there lived a young app developer named Andrew. By day, he fixed bugs and pushed updates. By night, he secretly believed the old stories his grandmother used to tell — the ones about hidden magic humming beneath ordinary things.

One evening, while debugging code at far too late an hour, Andrew noticed something strange on his screen. Between lines of perfectly normal Python, a message appeared:

// if you can read this, follow the blue cursor

Andrew blinked. “That… was not in the repo.”

The cursor began to move on its own, sliding across his monitor like it had somewhere very important to be. Against his better judgment (and every cybersecurity instinct he possessed), Andrew followed it.

Click.

Suddenly his apartment filled with a soft electric glow. His mechanical keyboard started typing by itself.

WELCOME, ANDREW. PATCH REQUIRED.

Now, Andrew had seen many weird production issues in his life, but none of them inv

Update the file `test.txt` in the repository using PyGitHub. This involves encoding the new content, creating a commit message, and using the `update_file` method provided by PyGitHub to update the file in the repository.

In [40]:
new_response = repo.update_file(fileInfo.path, new_content, "Update README.md file via API", fileInfo.sha)
print(new_response)

{'commit': Commit(sha="3573eb140b6a8e8f108941a0dd2dda071b89cf27"), 'content': ContentFile(path="test.txt")}
